# 00 — Load & assemble

**Plain English:** We read the real Freddie Mac loan files for three origination
years (2007, 2008, 2015), label every loan-month with a *delinquency bucket* and
an *IFRS 9 stage*, and run a data-quality check. This is the foundation the whole
monitor sits on.

**One-line terms**
- **Origination file** — one row per loan, its characteristics at the start.
- **Monthly performance (servicing) file** — one row per loan *per month*; carries the delinquency status that everything else is built from.
- **Delinquency bucket** — Current (0 days past due) / 30 / 60 / 90+ / Default (credit event) / Prepaid (paid off).
- **IFRS 9 stage** — Stage 1 performing · Stage 2 significant increase in credit risk (30+ days past due backstop) · Stage 3 credit-impaired/default (90+ days, or a credit event).

> Demonstrated on real loan-level mortgage data; the same monitoring *mechanics*
> apply to any commercial loan portfolio with a monthly status feed.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)

monitor library loaded — vintages: ['2007', '2008', '2015']


In [2]:
# Load each vintage and summarise its size + bucket mix --------------------
rows = []
for v in m.VINTAGES:
    orig = m.load_orig(v)
    svcg = m.load_svcg(v)
    svcg["bucket"] = m.dlq_to_bucket(svcg["dlq_status"])
    rows.append({
        "vintage": v,
        "loans (orig)": len(orig),
        "loan-months (svcg)": len(svcg),
        "months covered": int(svcg["loan_age"].max()) if svcg["loan_age"].notna().any() else None,
        "% ever 90+/default": round(
            100 * svcg.loc[svcg.bucket.isin(["90+", "Default"]), "loan_seq"].nunique() / len(orig), 2),
        "% rows status unknown": round(100 * svcg["bucket"].isna().mean(), 3),
    })
dq = pd.DataFrame(rows)
dq

D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\src\monitor.py:194: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\src\monitor.py:194: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


,vintage,loans (orig),loan-months (svcg),months covered,% ever 90+/default,% rows status unknown
0,2007,50000,3003932,225,16.47,0.0
1,2008,50000,2449679,212,9.30,0.0
2,2015,50000,3396519,128,4.06,0.0


In [3]:
# One clean results table for this notebook --------------------------------
dq.to_csv(m.OUT_TABLES / "00_data_quality.csv", index=False)
print("saved ->", m.OUT_TABLES / "00_data_quality.csv")

saved -> D:\Jane\Job Search\Github\bank\github project\mortgage-portfolio-monitoring\outputs\tables\00_data_quality.csv
